# Quantum Fourier transform and phase estimation

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import qiskit

from IPython.display import display
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.circuit.library import PhaseGate, QFTGate, XGate
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Operator, Statevector
from numpy.testing import assert_allclose

print("Qiskit", qiskit.__version__)

Qiskit 2.6.1


### Some coventions

For $n$ qubits, write the integer $x=\sum_{j=0}^{n-1}2^j x_j$ as the ket $|x\rangle=|x_{n-1}\rangle\otimes\cdots\otimes|x_0\rangle.$
NOte qiskit draws `q[0]` at the top. `q[0]` carries weight $2^0$. The rightmost character in a displayed bit string is bit 0. The amplitude of $|x\rangle$ is stored at index $x$ in `Statevector.data`.

In [3]:
# Example. 
# For the three-qubit state $|011\rangle$, flip qubits 0 and 1. 
# The only nonzero amplitude is at index $3$. 
# The three-qubit examples below use this convention even when the diagram reads from the top wire down.

basis_example = QuantumCircuit(3)
basis_example.x(0)
basis_example.x(1)
display(basis_example.draw("text"))

basis_state = Statevector.from_instruction(basis_example)
print(basis_state.to_dict())
assert_allclose(basis_state.data, np.eye(8)[3])

┌───┐
q_0: ┤ X ├
     ├───┤
q_1: ┤ X ├
     └───┘
q_2: ─────

{np.str_('011'): np.complex128(1+0j)}


## Classical and quantum Fourier transform

For $x=(x_j)\in \mathbb{R}^N$, $N=2^n$, the Fourier transform is the linear operator $\mathcal(F),\ \mathcal{F}x = y$ given by
$$
\mathcal{F} (x_j) = (y_k), \  y_k = \frac{1}{\sqrt{N}}\sum_{j=0}^{N-1} x_j e^{2\pi i j/N}
$$ 

With $\omega=e^{2\pi i/N}$, (with above positive-exponent convention), define the Fourier transform of a basis ket $|x\rangle$ similarly: 
$$
F_N|x\rangle=\frac1{\sqrt N}\sum_{y=0}^{N-1}\omega^{xy}|y\rangle,\qquad
(F_N)_{yx}=\frac1{\sqrt N}\omega^{xy}.
$$
using that coefficient of $|y\rangle$ represents the $y$-component. 

The inner product of $x$ and $x'$ is
$
\frac1N\sum_{y=0}^{N-1}(\omega^{x'-x})^y=\delta_{xx'}
$: if $x=x'$, every summand is 1. Otherwise the ratio $r=\omega^{x'-x}$ obeys $r\ne1$ and $r^N=1$, so the geometric sum is $(1-r^N)/(1-r)=0$. $F_N$ is unitary. Its inverse is the conjugate transpose. Note: numpy's orthonormal `ifft` uses positive sign and its orthonormal `fft` uses the negative sign.

### Factor the output over qubits

Write $y=\sum_{k=0}^{n-1}2^k y_k$. Then
$$\omega^{xy}=\prod_{k=0}^{n-1}\left(e^{2\pi i x/2^{n-k}}\right)^{y_k}.$$
The sum over $y$ is a sum over independent output bits $y_k$. Distributing the product gives
$$
F_N|x\rangle
=\bigotimes_{k=n-1,\ldots,0}
\frac{|0\rangle_k+e^{2\pi i x/2^{n-k}}|1\rangle_k}{\sqrt2}.
$$
Only the fractional part of $x/2^{n-k}$ matters. In binary this is
$$0.x_{n-k-1}\cdots x_0=\sum_{\ell=0}^{n-k-1}\frac{x_\ell}{2^{n-k-\ell}}.$$
For $x=3$ on three qubits, the phase fractions on qubits $(q_2,q_1,q_0)$ are $(1/2,\,3/4,\,3/8)$. The three factors have relative phases $(-1,\,-i,\,e^{3\pi i/4})$. Factoring on computational-basis inputs determines the circuit on every superposition by linearity.

In [3]:
def fourier_matrix(n):
    # np.outer makes entry (y, x) equal to y*x.
    size = 2**n
    indices = np.arange(size)
    return np.exp(2j * np.pi * np.outer(indices, indices) / size) / np.sqrt(size)


n = 3
size = 2**n
fourier = fourier_matrix(n)
assert_allclose(fourier.conj().T @ fourier, np.eye(size), atol=1e-12)
assert_allclose(fourier @ basis_state.data, np.fft.ifft(basis_state.data, norm="ortho"))

factors = [
    np.array([1, np.exp(2j * np.pi * fraction)]) / np.sqrt(2)
    for fraction in (1/2, 3/4, 3/8)
]
assert_allclose(np.kron(np.kron(factors[0], factors[1]), factors[2]), fourier[:, 3])